# 1. Problem framing and controlled-generator validation

## Goal

Can observable culture trajectories support a useful digital twin in a bounded synthetic system, before asking any learner to infer hidden state?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_fixed_environment_validation as fixed

cfg = fixed.Config()
print("culture seeds:", cfg.dataset_seeds)
print("model seeds:", cfg.model_seeds)
print("observable inputs:", ["temperature", "pH", "DO", "biomass", "product"])
print("lockbox quantities: synthetic latent state and generator internals")


## 2. Data generator

The fixed-environment generator simulates complete cultures under temperature, pH, and dissolved-oxygen conditions. It emits observable biomass/product trajectories and retains synthetic latent state only as a lockbox evaluation target.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# This is the explicit generator hand-off: a culture-level design becomes trajectories.
# `fast=True` is for notebook inspection only; it does not overwrite cached evidence.
traj, parameters, manifest = fixed.generate_dataset(cfg.dataset_seeds[0], cfg)
show(traj)


## 3. Model setup and training contract

No learned model is the primary object here. The experiment first verifies that the generator has nontrivial trajectories, environment effects, and declared failure worlds.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('metrics', 'data/canonical_metrics.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

There is no strain ranking in this phase. The screen is a generator audit: does each world meet its declared trajectory and split checks?

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
metrics = pd.read_csv(artifact('data/canonical_metrics.csv'))
show(metrics)


## 5. Matched comparison

Compare the three generator worlds and their audit metrics, rather than comparing a learner to an oracle.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = metrics.groupby('experiment', dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = metrics.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

A passing audit makes the synthetic benchmark usable; it does not establish biological realism or wet-lab validity.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    fixed.run_all(fast=False, write=True)
